# Vehicle Type Classification - Project-I Starter

Competition: `vehicle-type-classification` | Target: `vehicle_class` (0=Sedan, 1=SUV, 2=Sports Car).
**Allowed models ONLY (classification_models.md): LDA and L2 Logistic/Softmax.**
Structure: 1 Loading, 2 EDA, 3 Cleaning, 4 Feature audit, 5 Preprocessing, 6 Reusable evaluation
functions, 7 Direct family comparison, 8 Hyper-parameter tuning, 9 Correlation-drop experiment,
10 intentionally OMITTED (ensembles are not Part-1 models and are banned by the rules),
11 Diagnostics, 12 Final CV of shortlisted models + submission.
`vehicle_id` is never a feature. Kernel: `ML Vehicle Type (.venv)`. No auto-submit.


### Sec.1 - Data Loading
Setup (versions, seed 42) then reads train/test/sample. Sample file defines the
exact submission format. On Kaggle a shim links /kaggle/input files here.


In [ ]:
# 1. Setup (.venv kernel, seed 42)
import sys, sklearn, pandas as pd, numpy as np
print(sys.version)
print('sklearn', sklearn.__version__)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# 1. Load data
train = pd.read_csv('vehicle_train_10000.csv')
test = pd.read_csv('vehicle_test.csv')
sample = pd.read_csv('vehicle_sample_submission.csv')
print(train.shape, test.shape, sample.shape)
display(train.head())
display(test.head())
display(sample.head())
print('Target values:', sorted(train['vehicle_class'].unique()))

### Sec.2 - Exploratory Data Analysis
Duplicates (~2500 feature-dupes), balanced classes (~33% each),
zero missing, wide scales and 0.95 correlations. Findings drive Sec.3-5.


In [ ]:
# 2. EDA: duplicates, class balance, missing, scales
print('duplicated rows (full):', train.duplicated().sum())
print(train['vehicle_class'].value_counts(normalize=True).round(3))
print('\nMissing train (%):')
print((train.isna().mean()*100).round(2).sort_values(ascending=False).head(10))
print('\nMissing test (%):')
print((test.isna().mean()*100).round(2).sort_values(ascending=False).head(10))
display(train.describe().T.head(25))

### Sec.3 - Data Cleaning
Audits (missing, full-row dups, id overlap) and the USE_DEDUP switch. Default
OFF to match the executed run; applied, if enabled, in Sec.5 where features are known.


In [ ]:
# 3. Data Cleaning: audits + optional dedup switch (default OFF = as-run on Kaggle)
train_clean = train.copy(); test_clean = test.copy()
print('missing train:', int(train_clean.isna().sum().sum()), '| test:', int(test_clean.isna().sum().sum()))
print('full-row dups train:', int(train_clean.duplicated().sum()))
print('train/test id overlap:', len(set(train_clean['vehicle_id']) & set(test_clean['vehicle_id'])))
USE_DEDUP = False  # set True to drop feature-duplicate rows (host's data-quality quirk)
print('USE_DEDUP =', USE_DEDUP, '(keep False to match the executed Kaggle run)')


### Sec.4 - Feature Engineering: audit (no new columns needed)
IDs fixed, feature inventory, scale
spread and >0.9 pair count - the evidence for StandardScaler and pooled/regularized models.


In [ ]:
# 4. Feature Engineering: inventory + scale/correlation audit (no new columns needed)
# All 20 inputs are numeric; vehicle_id is an identifier. Audit motivates StandardScaler (wide scale
# ranges) and pooled/regularized models (0.95 correlations).
ID, TARGET = 'vehicle_id', 'vehicle_class'
features = [c for c in train_clean.columns if c not in (ID, TARGET)]
print('n features:', len(features))
_std = train_clean[features].std().sort_values()
print('smallest std:', _std.head(3).round(3).to_dict())
print('largest std :', _std.tail(3).round(1).to_dict())
_c = train_clean[features].corr().abs()
import numpy as np
_triu = _c.where(np.triu(np.ones(_c.shape), k=1).astype(bool)).stack()
print('pairs with |corr| > 0.9:', int((_triu > 0.9).sum()))
print('top pair:', _triu.sort_values(ascending=False).head(1).round(3).to_dict())


### Sec.5 - Data Preprocessing (fit on train only)
Median-impute + StandardScaler pipeline on the 20
numeric features; statistics learned from training folds only.


In [ ]:
# 5. Preprocessing pipeline (fit on train only)
# All 20 features are numeric. vehicle_id is identifier only.
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

print(len(features), features)
prep = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
X = train_clean[features]
y = train_clean[TARGET]
X_test = test_clean[features]
# Optional: drop exact duplicates for training (keep one copy)
# X, y dedup: uncomment if it helps validation
# dedup_mask = ~pd.concat([X, y], axis=1).duplicated()
# X, y = X[dedup_mask], y[dedup_mask]
# print('after dedup:', X.shape)

In [ ]:
# 5. Optional dedup (features known here)
if USE_DEDUP:
    _keep = ~pd.concat([X, y], axis=1).duplicated(subset=features, keep='first')
    X, y = X[_keep].reset_index(drop=True), y[_keep].reset_index(drop=True)
    print('after dedup:', X.shape)
else:
    print('dedup skipped; training rows:', X.shape)


### Sec.6 - Model Evaluation: split + reusable functions
Stratified 80/20 split (competition metric =
accuracy) and acc_summary() used by every later section.


In [ ]:
# 6. Stratified train/val split (accuracy)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(Xtr.shape, Xva.shape)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# 6b. Reusable evaluation functions (used in Sec.7-12)
from sklearn.metrics import accuracy_score, classification_report
def acc_summary(name, yt, yp):
    print(f'{name:12s} acc {accuracy_score(yt, yp):.4f}')
    print(classification_report(yt, yp, target_names=['Sedan', 'SUV', 'Sports Car'], digits=4))
    return float(accuracy_score(yt, yp))


### Sec.7 - Model Family Direct Comparisons
One fit per allowed model at default settings (LDA pair +
Logistic C=1.0) with table and bar chart - answers 'which family first' before tuning.


In [ ]:
# 7a. LDA plain vs shrinkage (one fit each)
# Different things tried here: plain SVD vs LSQR with automatic shrinkage (same LDA model).
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
pipe_lda = Pipeline([('prep', prep), ('model', LinearDiscriminantAnalysis())])
pipe_lda_s = Pipeline([('prep', prep),
    ('model', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))])
pipe_lda.fit(Xtr, ytr)
pipe_lda_s.fit(Xtr, ytr)
print('LDA val acc:', round(accuracy_score(yva, pipe_lda.predict(Xva)), 4))
print('LDA-shrink val acc:', round(accuracy_score(yva, pipe_lda_s.predict(Xva)), 4))


In [ ]:
# Sec.7a plots: confusion matrices LDA vs LDA-shrinkage
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
LABELS = ['Sedan', 'SUV', 'Sports Car']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(yva, pipe_lda.predict(Xva), display_labels=LABELS, ax=ax[0])
ax[0].set_title('LDA (plain)')
ConfusionMatrixDisplay.from_predictions(yva, pipe_lda_s.predict(Xva), display_labels=LABELS, ax=ax[1])
ax[1].set_title('LDA (shrinkage=auto)')
plt.tight_layout(); plt.savefig('plots/lda_confusion.png'); plt.show()
print('LDA val acc:', round(accuracy_score(yva, pipe_lda.predict(Xva)), 4))
print('LDA-shrink val acc:', round(accuracy_score(yva, pipe_lda_s.predict(Xva)), 4))


In [ ]:
# 7b. Logistic default (C=1.0) for the direct comparison (tuning comes in Sec.8)
from sklearn.linear_model import LogisticRegression
pipe_log_d = Pipeline([('prep', prep),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
pipe_log_d.fit(Xtr, ytr)
print('Logistic(C=1.0) val acc:', round(accuracy_score(yva, pipe_log_d.predict(Xva)), 4))


In [ ]:
# 7c. Direct-comparison table + bar chart (one fit per allowed model)
import matplotlib.pyplot as plt
_res = {
    'LDA': accuracy_score(yva, pipe_lda.predict(Xva)),
    'LDA-shrink': accuracy_score(yva, pipe_lda_s.predict(Xva)),
    'Logistic': accuracy_score(yva, pipe_log_d.predict(Xva)),
}
for _k, _v in sorted(_res.items(), key=lambda kv: -kv[1]):
    print(f'{_k:12s} {_v:.4f}')
plt.figure()
plt.bar(list(_res.keys()), list(_res.values()))
plt.ylabel('val accuracy (higher is better)')
plt.title('Sec.7: one-fit-per-family comparison')
import os; os.makedirs('plots', exist_ok=True)
plt.savefig('plots/sec7_compare.png'); plt.show()


### Sec.8 - Hyper-parameter Tuning: Logistic C grid
Wide C grid (0.01-1000) via stratified 5-fold CV +
U-curve. LDA needs no grid (its shrinkage variant was compared in Sec.7).


In [ ]:
# 8. Logistic C grid (stratified 5-fold CV)
from sklearn.linear_model import LogisticRegression
pipe_log = Pipeline([('prep', prep),
    ('model', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000, random_state=RANDOM_STATE))])
grid_log = GridSearchCV(pipe_log, {'model__C': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}, cv=cv)
grid_log.fit(Xtr, ytr)
print('best:', grid_log.best_params_)
print('val acc:', accuracy_score(yva, grid_log.predict(Xva)))

In [ ]:
# Sec.8 plot: C vs CV accuracy (Logistic/Softmax)
import matplotlib.pyplot as plt
import pandas as pd
r = pd.DataFrame(grid_log.cv_results_)
a = pd.to_numeric(r['param_model__C'])
s = r['mean_test_score']
order = a.sort_values().index
plt.figure()
plt.semilogx(a[order], s[order], marker='o')
plt.xlabel('C'); plt.ylabel('CV accuracy (higher is better)')
plt.title('Logistic: C vs CV accuracy')
plt.savefig('plots/logistic_C.png'); plt.show()
print('best:', grid_log.best_params_, '| val acc:', round(accuracy_score(yva, grid_log.predict(Xva)), 4))


### Sec.9 - Experiment: drop highly-correlated features
Same LDA/Logistic on 16 instead of 20 columns
(one of each >0.9 pair removed). Keeps or drops per YOUR numbers - the decision goes to the report.


In [ ]:
# 9. Experiment: drop highly-correlated features (same allowed models, fewer columns)
# Drops one of each >0.9 pair (keep torque_nm, wheelbase_mm). Question: does de-duplicating the
# 0.95-correlated engine/size block help LDA/Logistic or remove signal?
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
DROP_CORR = ['horsepower_hp', 'engine_displacement_l', 'vehicle_width_mm', 'vehicle_length_mm']
_f2 = [c for c in features if c not in DROP_CORR]
_lda2 = Pipeline([('prep', prep), ('model', LinearDiscriminantAnalysis())]).fit(Xtr[_f2], ytr)
_log2 = Pipeline([('prep', prep), ('model', LogisticRegression(C=1.0, max_iter=2000, random_state=RANDOM_STATE))]).fit(Xtr[_f2], ytr)
print('features kept:', len(_f2), '| dropped:', DROP_CORR)
print('LDA full vs reduced     :', round(accuracy_score(yva, pipe_lda.predict(Xva)), 4), 'vs', round(accuracy_score(yva, _lda2.predict(Xva[_f2])), 4))
print('Logistic full vs reduced:', round(accuracy_score(yva, pipe_log_d.predict(Xva)), 4), 'vs', round(accuracy_score(yva, _log2.predict(Xva[_f2])), 4))
print('Conclusion for the report: keep or drop based on YOUR numbers above.')


### Sec.10 - OMITTED by design
Ensembles (e.g. Random Forest) are NOT Part-1 models, so the competition
rules ban them. This section is intentionally empty.


### Sec.11 - Model Diagnostics
Per-class precision/recall/F1 plus confusion of the tuned winner: the
lowest-F1 class and biggest off-diagonal cell are your report's error analysis.


In [ ]:
# 11. Model Diagnostics: per-class report + confusion of the tuned winner
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
print(classification_report(yva, grid_log.predict(Xva), target_names=['Sedan', 'SUV', 'Sports Car'], digits=4))
ConfusionMatrixDisplay.from_predictions(yva, grid_log.predict(Xva), display_labels=['Sedan', 'SUV', 'Sports Car'])
plt.title('Tuned Logistic: confusion on validation (rows=true, cols=pred)')
import os; os.makedirs('plots', exist_ok=True)
plt.savefig('plots/tuned_confusion.png'); plt.show()
print('Read it: lowest-F1 class and biggest off-diagonal cell name the error pattern for the report.')


### Sec.12 - Cross-Validation of Shortlisted Models + submission
Honest 5-fold CV on FULL train
(mean +- std bars), then refit the chosen winner and write submission.csv for MANUAL upload.


In [ ]:
# 12. Final 5-fold CV of SHORTLISTED models on FULL train + error bars
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
_cands = {'LDA': pipe_lda, 'LDA-shrink': pipe_lda_s, 'Logistic': grid_log.best_estimator_}
_names, _means, _stds = [], [], []
for _n, _m in _cands.items():
    _s = cross_val_score(_m, X, y, cv=5, scoring='accuracy')
    _names.append(_n); _means.append(float(_s.mean())); _stds.append(float(_s.std()))
    print(f'{_n:12s} CV acc {_s.mean():.4f} +- {_s.std():.4f}')
plt.figure(); plt.errorbar(_names, _means, yerr=_stds, fmt='o', capsize=6)
plt.ylabel('CV accuracy (higher is better)'); plt.title('Sec.12: shortlisted 5-fold CV (mean +- std)')
import os; os.makedirs('plots', exist_ok=True)
plt.savefig('plots/sec12_cv.png'); plt.show()


In [ ]:
# 12b. Refit winner on FULL train -> submission.csv
best_model = grid_log.best_estimator_  # <-- change to winner from section 6
best_model.fit(X, y)
preds = best_model.predict(X_test).astype(int)
sub = __import__('pandas').DataFrame({ID: test[ID], TARGET: preds})
assert sub.shape[0] == len(test) and list(sub.columns) == list(sample.columns)
assert set(sub[TARGET].unique()) <= {0, 1, 2}
sub.to_csv('submission.csv', index=False)
print(sub.head(), sub.shape)
print('Saved submission.csv - upload MANUALLY to Kaggle. This notebook never submits via API.')